In [2]:
from typing_extensions import TypedDict
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
import json
import nest_asyncio
from IPython.display import display,Image
from typing import Sequence
import time
import shutil
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, HumanMessage
from typing import Annotated
from datetime import datetime
from langgraph.types import Send
import asyncio
import edge_tts
from moviepy import *
import random
import logging
from ffmpeg import FFmpeg, Progress 
import cv2
import numpy as np
import os
import requests
from PIL import Image, ImageDraw, ImageFont
nest_asyncio.apply()

from dotenv import load_dotenv,find_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
import os
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
   raise ValueError('GEMINI_API_KEY is not set in the env file')
print('GEMINI_API_KEY loaded successfully')    

llm = ChatGoogleGenerativeAI(api_key=GEMINI_API_KEY,model='gemini-1.5-flash')


 
    
class MainCharacters(TypedDict):
    name: str
    appearance: str
    characteristics: str
    
class SupportingCharacters(TypedDict):
    name: str
    appearance: str
    characteristics: str
    
class ScenesList(TypedDict):
    id: str
    scene: str
    description: str
    narration: str
    img_prompt: str
    object_description:str
    save_audio_path:str
    save_image_path:str  
    audio_duration:str

    
class GraphState(TypedDict): 
    story_theme: str
    scene_list: list[ScenesList] 
    supporting_characters: list[SupportingCharacters]
    main_characters:list[MainCharacters] 
    pre_processing_video_path:str
    combined_audio_path:str
    voices_folder:str
    images_folder:str
    messages: Annotated[Sequence[BaseMessage], add_messages]
    
class SubState(TypedDict): 
    current_scene: ScenesList
    output_folder: str
    

def generate_story(state: GraphState) -> GraphState:
    story_theme = state["story_theme"]

    new_prompt = """Based on the given story theme, generate a structured list of  scenes where the total narration duration does not exceed 50 seconds.
    and  create a brief description of the main and supporting character, object, or scene. Include specific details about appearance, characteristics . 
    This description will be used to maintain consistency across multiple scenes.
    Each scene must include an **`Object_Description`** field, which provides a short description of key objects, scenery, or important elements in that scene.  
    and also generate a narration that exactly follows this text, starting with 'Once upon a time...' 

      ### Story Theme:
      {story_theme}

      ### Output Format (JSON):
      {{
        "main_characters": [
              {{
                "name": "",
                "appearance": "",
                "characteristics": ""
              }}
            ],
            "supporting_characters": [
              {{
                "name": "",
                "appearance": "",
                "characteristics": ""
              }}
            ],
          "scenes": [
            {{
              "id": "1",
              "scene": "Engaging Beginning",
              "description": "Begin with a captivating moment to grab children's attention.",
              "narration": ""
              "object_description": ""
            }},
         
          ],
          
        }}

      - **Generate  scenes** to ensure a smooth and structured story.
      - **Don't generate more than 5 scenes.**
      - **Start with an engaging scene** to hook the audience immediately. 
      - **Write concise, engaging, and clear narration** to fit within 50 seconds.
      - **End with a meaningful but natural lesson** without making it feel forced.
      - **Use simple and engaging language** suitable for children.
      - **Ensure smooth transitions** so the story flows naturally.

      Return ONLY valid JSON output without any extra formatting or explanations. Do NOT use markdown formatting (e.g., no triple backticks).
      Strictly output **only JSON** without extra text."""

    sys_msg = SystemMessage(
        content="You are an assistant that extracts structured details from a story."
    )
    hum_msg = HumanMessage(content=new_prompt.format(story_theme=story_theme))

    start_time = time.time()
    res = llm.invoke([sys_msg, hum_msg])
    end_time = time.time()

    print(f"Time taken to generate story characters: {end_time - start_time} seconds")
    print(f"llm result :{res}") 
    raw_content = res.content.strip()
    if raw_content.startswith("```") and raw_content.endswith("```"):
        raw_content = raw_content.strip("`")
        if raw_content.lower().startswith('json'):
            raw_content = raw_content[4:].strip() 
    
    try:
        parsed_resp = json.loads(raw_content)
    except json.JSONDecodeError as e:
        raise ValueError(f"Error parsing JSON: {e}\nRaw content: {raw_content}")

    for scene in parsed_resp["scenes"]:
        prompt_template = f"""Create a detailed, photorealistic image of the following scene:
        {scene["description"]}
        
        **Main Characters**:
        {", ".join([f'{char.get("name",'')} - {char.get("appearance",'')}, {char.get("characteristics",'')}' for char in parsed_resp['main_characters'] if parsed_resp['main_characters']])}
 

        **Supporting Characters**:
        {", ".join([f'{supchar.get("name","")} - {supchar.get("appearance","")} - {supchar.get("characteristics","")}' for supchar in parsed_resp['supporting_characters'] if parsed_resp['supporting_characters']])}
        
        **Objects**:
        {scene["object_description"]}
        **Mood & Lighting**: Cinematic, immersive atmosphere with realistic lighting to match the scene's emotions.

        The illustration should capture the story’s essence and atmosphere."""
        scene["img_prompt"] = prompt_template
 
    state["scene_list"] = parsed_resp["scenes"]
 
    state["supporting_characters"] = parsed_resp.get("supporting_characters", [])
    state["main_characters"] = parsed_resp.get("main_characters", [])
    return state

    
    
async def save_voice(save_path, current_voice):
    await current_voice.save(save_path)


async def call_llm_gen_voice(voice):
    try:
        tts = edge_tts.Communicate(
            voice, voice="en-US-JennyNeural", volume="+100%", pitch="+5Hz"
        )
        return tts
    except Exception as e:
        raise ValueError("Error while generating the voice", e)


async def generate_scene_voice(state: SubState):
    cur_scene = state["current_scene"]
    scene_id = int(cur_scene["id"])
    output_folder = state["output_folder"]
    narration_text = cur_scene.get("narration", "")

    try:
        print(f"Generating voice for scene {scene_id}")
        generated_voice = await call_llm_gen_voice(narration_text)
        save_path = os.path.join(output_folder, f"voice_scene_{scene_id}.mp3")
        save_time = time.time()

        await save_voice(save_path, generated_voice)
        end_save_time = time.time()
        
        print(f"saving time for this scene {scene_id}={end_save_time-save_time}")
        print(f"Generated voice successfully for scene {scene_id}")
        try:
            with AudioFileClip(save_path) as clip:
                audio_duration = clip.duration
        except Exception as e:
            print(f"Error generating voice for scene {scene_id}: {e}")

        message = HumanMessage(
            content="",
            additional_kwargs={
                "voice_id": scene_id,
                "save_path": save_path,
                "audio_duration": audio_duration
            }
        )
        
        return {"messages": [message]}

    except Exception as e:
        print(f"Error generating voice for scene {scene_id}: {e}")


def continue_generate_voice(state: GraphState):
    scene_list = state["scene_list"]
    time_stamp = datetime.now().strftime("%Y%m%d%H%M%S")
    dynamic_folder = f"voices_{time_stamp}"
    output_folder = os.path.join("Generated_voices", dynamic_folder)
    output_folder = os.path.join("final_output", output_folder)
    os.makedirs(output_folder, exist_ok=True)  
    return [
        Send(
            "generate_scene_voice",
            {
                "current_scene": scene,
                "output_folder": output_folder,
            },
        )
        for scene in scene_list
    ]

def sync_generate_scene_voice(state: SubState):
    return asyncio.run(generate_scene_voice(state))






def saveImage(image_content, save_path):
    output_folder = os.path.dirname(save_path)
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    with open(save_path, "wb") as f:
        f.write(image_content)


def images_generates(prompt: str):
    try: 
        color_value1 = format(random.randint(0, 16777215), '06x')  
        color_value2 = format(random.randint(0, 16777215), '06x')
        image_url = f"https://placehold.co/1080x1920/{color_value1}/{color_value2}/png" 
        return image_url 

    except Exception as e:
        logging.error(f"Unexpected error generating image: {e}")
        raise


 

def generate_scence_image(state:SubState):
    cur_scene = state["current_scene"]
    scene_id = int(cur_scene["id"])
    output_folder = state["output_folder"]  
    image_prompt = cur_scene.get("img_prompt", "")
    try:
        print(f"Generating image for scene {scene_id}")
        imageUrl = images_generates(image_prompt)
        if not imageUrl:
            raise ValueError("Generated image URL is empty.")
        imageContent = requests.get(imageUrl).content
        if not imageContent:
            raise ValueError("Failed to retrieve image content.")
        save_path = os.path.join(output_folder, f"image_scene_{scene_id}.png")
        saveImage(imageContent, save_path)
        print(f"Images saved to {save_path}")
        message = HumanMessage(content='',additional_kwargs={"image_id": scene_id,"save_path": save_path})
        return {'messages':[message]}

    except Exception as e:
        logging.error(f"Error processing scene {scene_id}: {e}")
       
 
     

def continue_scence_image(state:GraphState):
    scene_list = state['scene_list']
    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    dynamic_folder = f"images_{timestamp}"
    output_folder = os.path.join("Generated_images", dynamic_folder)
    output_folder = os.path.join("final_output", output_folder)
    os.makedirs(output_folder, exist_ok=True) 
    return [Send("generate_scence_image",{'current_scene':value, 'output_folder':output_folder}) for value in scene_list ]
     
     
     
     
     






def final_node(state: GraphState):
    total_messages = state["messages"]
    print(f"Toatal messages:{total_messages}")
    scene_list = state["scene_list"]
    list_images,list_voices =set(),set()
    for scene in scene_list:
        get_id = int(scene["id"])
     
        for msg in total_messages:
            image_id = msg.additional_kwargs.get('image_id')
            voice_id = msg.additional_kwargs.get("voice_id")
            
            if voice_id is not None and get_id == voice_id:
                aud_dur = msg.additional_kwargs.get('audio_duration','')
                save_path = msg.additional_kwargs.get('save_path','')
                print(f"{get_id} {msg}")
                scene["save_audio_path"] = save_path
                scene["audio_duration"] = aud_dur
                list_voices.add(os.path.dirname(save_path))
            if get_id== image_id : 
                save_path = msg.additional_kwargs.get('save_path','')
                scene['save_image_path'] = msg.additional_kwargs.get('save_path','')
                list_images.add(os.path.dirname(save_path))
    state['images_folder'] =list(list_images)[0]  if list_images else ""      
    state['voices_folder'] =list(list_voices)[0]  if list_voices else ""      
    return state
def split_text_into_segments(text, font, max_width):
    """Split the text into segments that fit within max_width."""
    words = text.split()
    segments = []
    current_segment = ""
    for word in words:
        test_line = current_segment + (" " if current_segment else "") + word
        w = font.getbbox(test_line)[2]
        if w <= max_width:
            current_segment = test_line
        else:
            if current_segment:
                segments.append(current_segment)
            current_segment = word
    if current_segment:
        segments.append(current_segment)
    print(f"print segments: {segments}")    
    return segments

def zoom_in(image, num_frames=30, zoom_factor=0.5):
    """Generates frames for a zoom-in effect."""
    frames = []
    h, w = image.shape[:2]
    for i in range(num_frames):
        scale = 1.0 + (i / num_frames) * zoom_factor
        center = (w // 2, h // 2)
        M = cv2.getRotationMatrix2D(center, 0, scale)
        frame = cv2.warpAffine(image, M, (w, h))
        frames.append(frame)
    return frames


def zoom_out(image, num_frames=30, zoom_factor=0.5):
    """Generates frames for a zoom-out effect without blinking."""
    frames = []
    h, w = image.shape[:2]
    for i in range(num_frames):
        scale = 1.0 + ((num_frames - i - 1) / num_frames) * zoom_factor
        center = (w // 2, h // 2)
        M = cv2.getRotationMatrix2D(center, 0, scale)
        frame = cv2.warpAffine(image, M, (w, h))
        frames.append(frame)
    return frames


def fade_in(image, num_frames=2):
    """Creates a fade-in effect from black to the image."""
    frames = []
    black = np.zeros_like(image)
    for i in range(num_frames):
        alpha = i / num_frames
        frame = cv2.addWeighted(image, alpha, black, 1 - alpha, 0)
        frames.append(frame)
    return frames


def fade_out(image, num_frames=30):
    """Creates a fade-out effect from image to black."""
    frames = []
    black = np.zeros_like(image)
    for i in range(num_frames):
        alpha = 1 - i / num_frames
        frame = cv2.addWeighted(image, alpha, black, 1 - alpha, 0)
        frames.append(frame)
    return frames
def pre_processing_video(state:GraphState):
    fps=10  ############################ Frame per second ##############
    
    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    dynamic_folder=f"pre_video_{timestamp}"
    output_folder=os.path.join("Pre_Generated_videos",dynamic_folder)
    output_folder=os.path.join("final_output",output_folder)
    os.makedirs(output_folder, exist_ok=True)
    file_name = f'pre_video_{timestamp}.mp4'
    
    pre_processing_video_path=os.path.join(output_folder,file_name)
    scene_list=state['scene_list']
    if not scene_list:
        raise ValueError("No scenes provided.")

    first_img = cv2.imread(scene_list[0]["save_image_path"])
    if first_img is None:
        raise ValueError(f"Cannot load image: {scene_list[0]['save_image_path']}")

    h, w, _ = first_img.shape
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    video_writer = cv2.VideoWriter(pre_processing_video_path, fourcc, fps, (w, h))
    caption_font = ImageFont.truetype("arial.ttf", 80)
    # Maximum width for caption text (with some margin)
    max_caption_width = w - 40
    
    starting_time = time.time()
    for idx, scene in enumerate(scene_list):
        img_path = scene["save_image_path"]
        duration = float(scene["audio_duration"])
        narration = scene["narration"]

        img = cv2.imread(img_path)
        if img is None:
            print(f"Warning: Cannot load image {img_path}. Skipping.")
            continue
        img = cv2.resize(img, (w, h))

        total_frames = int(fps * duration)
        fade_in_frame = int(fps * 1)
        fade_out_frame = int(fps * 1)
        first_image_duration = total_frames - fade_out_frame

        if idx == 0:
            zoom_in_frames = zoom_in(img, num_frames=first_image_duration, zoom_factor=0.5)
            final_zoom_frame = zoom_in_frames[-1]
            fade_out_frames = fade_out(final_zoom_frame, num_frames=fade_out_frame)
            effect_frames = zoom_in_frames + fade_out_frames
        elif idx % 2 == 1:
            zoom_out_frames = zoom_out(img, num_frames=first_image_duration, zoom_factor=0.5)
            final_zoom_frame = zoom_out_frames[-1]
            fade_out_frames = fade_out(final_zoom_frame, num_frames=fade_out_frame)
            effect_frames = zoom_out_frames + fade_out_frames
        else:
            fade_in_frames = fade_in(img, num_frames=fade_in_frame)
            zoom_in_frames = zoom_in(img, num_frames=first_image_duration, zoom_factor=0.5)
            final_zoom_frame = zoom_in_frames[-1]
            fade_out_frames = fade_out(final_zoom_frame, num_frames=fade_out_frame)
            effect_frames = fade_in_frames + zoom_in_frames + fade_out_frames
        segments = split_text_into_segments(narration, caption_font, max_caption_width)
        num_segments = len(segments)
        if num_segments == 0:
            segments = [""]
         
        total_chars = sum(len(seg) for seg in segments)
        global_frame_index = 0
        for segment in segments:
            seg_chars = len(segment) 
            segment_frames = int((seg_chars / total_chars) * total_frames) 
            segment_frames = max(segment_frames, 1)
            
            for i in range(segment_frames): 
                if global_frame_index >= len(effect_frames):
                    break
                frame = effect_frames[global_frame_index].copy()
                frame_pil = Image.fromarray(frame)
                draw = ImageDraw.Draw(frame_pil) 
                char_count = int(((i + 1) / segment_frames) * seg_chars)
                displayed_text = segment[:char_count] 
                text_width, text_height = caption_font.getbbox(displayed_text)[2:4]
                x = (w - text_width) // 2
                y = (h - text_height) // 2  
                padding = 10
                channel_font = ImageFont.truetype("arial.ttf", 16)
                draw.rectangle([(x - padding, y - padding), (x + text_width + padding, y + text_height + padding)], fill=(0, 0, 0))
               
                draw.text((x, y), displayed_text, font=caption_font, fill=(255, 255, 255))
                channel_text = "Youtube"
                ch_text_width, ch_text_height = channel_font.getbbox(channel_text)[2:4]
                x_channel = w - ch_text_width - 20   
                y_channel = 20  
                draw.text((x_channel, y_channel), channel_text, font=channel_font, fill=(255, 255, 255))
                
                video_writer.write(np.array(frame_pil))
                global_frame_index += 1
        # for frame in effect_frames:
        #     text = "Youtube"
        #     font = cv2.FONT_HERSHEY_SIMPLEX
        #     font_scale = 1
        #     font_thickness = 2
        #     color = (220, 255, 255)
        #     text_size = cv2.getTextSize(text, font, font_scale, font_thickness)[0]
        #     text_x = w - text_size[0] - 20
        #     text_y = 60
        #     cv2.putText(frame, text, (text_x, text_y), font, font_scale, color, font_thickness, cv2.LINE_AA)

        #     video_writer.write(frame)

    video_writer.release()
    ending_time = time.time()
    print(f'Total time it takes to convert images into video frames: {ending_time -starting_time}')
    
    print(f"pre_processing_video saved as {pre_processing_video_path}")     
  
    return {'pre_processing_video_path':pre_processing_video_path}
from pydub import AudioSegment
def combining_audio(state:GraphState):
    scene_list=state['scene_list']
    voice_clips = []
    for voice in scene_list:
        print(f'combining _audio {voice}')
        audio_path=voice['save_audio_path']
        if not os.path.exists(audio_path):
            print(f"Warning: Audio file not found:{audio_path}. Skipping")
            continue
        try: 
            voice_clips.append(AudioFileClip(audio_path))
        except Exception as e:
            print(f"Error loading {audio_path}:{e}")    
    if not voice_clips:
        raise ValueError("No Valid audio files to combine")
    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    dynamic_folder=f"audio_{timestamp}"
    output_folder=os.path.join("Combined_audio",dynamic_folder)
    output_folder=os.path.join("final_output",output_folder)
    os.makedirs(output_folder, exist_ok=True)
    file_name = f'audio_{timestamp}.mp3'
    final_path=os.path.join(output_folder,file_name)
    
     
    with concatenate_audioclips(voice_clips) as audio:
        audio.write_audiofile(final_path)
    narration = AudioSegment.from_file(final_path)
    bg_music = AudioSegment.from_file("bg_music.mp3")
    bg_music = bg_music - 45
    while len(bg_music) < len(narration):
        bg_music += bg_music    
    bg_music = bg_music[:len(narration)] 
    merged_audio = narration.overlay(bg_music)
    merged_audio.export(final_path, format="mp3")
    print(f"Combined audio saved as {final_path}")  
    return {'combined_audio_path':final_path}
 

def finaliaz_video(state: GraphState):
    time_stamp = datetime.now().strftime("%Y%m%d%H%M%S")
    video_path = state['pre_processing_video_path']  
    audio_path = state['combined_audio_path']
    images_folder = state['images_folder']
    voices_folder = state['voices_folder']
    
    output_path = f"./final_output/final_video_{time_stamp}.mp4" 
    print('Merging video and audio') 
    print(f'video_path: {video_path}, ')
    print(f'audio_path :  ,{audio_path}, ')
    print(f'images_folder:  {images_folder}')
    print(f'voices_folder : {voices_folder}')
    !ffmpeg -i "{video_path}" -i "{audio_path}" -c:v copy -c:a aac -strict experimental "{output_path}"
    
     
    def delete_file_and_parent(file_path):
        if os.path.exists(file_path):
            try:
                os.remove(file_path)
                print(f"Deleted file: {file_path}")
            except PermissionError as e:
                print(f"PermissionError: {e} for file {file_path}")
            parent_folder = os.path.dirname(file_path)
            if os.path.exists(parent_folder) and not os.listdir(parent_folder):
                try:
                    shutil.rmtree(parent_folder)
                    print(f"Deleted empty folder: {parent_folder}")
                except Exception as e:
                    print(f"Error deleting folder {parent_folder}: {e}")
    
    def delete_folder(folder_path):
        if folder_path and os.path.exists(folder_path):
            try:
                shutil.rmtree(folder_path)
                print(f"Deleted folder: {folder_path}")
            except Exception as e:
                print(f"Error deleting folder {folder_path}: {e}")
    
   
     
    # delete_file_and_parent(video_path)
    # delete_file_and_parent(audio_path)
    # delete_folder(voices_folder)     
    # delete_folder(images_folder)
    print("Merged video and audio successfully and output path is ", output_path)
    return state

workflow = StateGraph(GraphState)
workflow.add_node('generate_story',generate_story) 
workflow.add_node("generate_scene_voice", sync_generate_scene_voice)
workflow.add_node('pre_processing_video',pre_processing_video)
workflow.add_node('generate_scence_image',generate_scence_image) 
workflow.add_node("combining_audio", combining_audio) 
workflow.add_node('final_node',final_node)
workflow.add_node('finaliaz_video',finaliaz_video)


workflow.add_edge(START,'generate_story') 
workflow.add_conditional_edges('generate_story', continue_generate_voice, ["generate_scene_voice"])
workflow.add_conditional_edges('generate_story',continue_scence_image,['generate_scence_image'])
workflow.add_edge(["generate_scene_voice","generate_scence_image"], "final_node")
workflow.add_edge('final_node','combining_audio') 
workflow.add_edge("final_node", 'pre_processing_video')
workflow.add_edge(['pre_processing_video','combining_audio'],'finaliaz_video') 
workflow.add_edge('finaliaz_video',END)
 
app = workflow.compile()
 
story_theme =  """Once, a rabbit mocked a slow-moving tortoise. The tortoise challenged him to a race. Confident of his speed, the rabbit dashed ahead and took a nap. Meanwhile, the tortoise kept moving steadily. By the time the rabbit woke up, the tortoise had already crossed the finish line.""" 
resp = app.invoke({'story_theme':story_theme})
print(resp)

GEMINI_API_KEY loaded successfully
Time taken to generate story characters: 4.720532417297363 seconds
llm result :content='```json\n{\n  "main_characters": [\n    {\n      "name": "Tortoise",\n      "appearance": "Slow-moving, with a hard shell, brown and green",\n      "characteristics": "Patient, persistent, determined"\n    },\n    {\n      "name": "Rabbit",\n      "appearance": "Fast, fluffy white fur, long ears",\n      "characteristics": "Overconfident, boastful, lazy"\n    }\n  ],\n  "supporting_characters": [],\n  "scenes": [\n    {\n      "id": "1",\n      "scene": "The Mocking",\n      "description": "Rabbit mocks the tortoise\'s slow pace.",\n      "narration": "Once upon a time, there was a speedy rabbit who mocked a slow tortoise.  \'You\'re so slow!\' he teased.",\n      "object_description": "A sunny forest path"\n    },\n    {\n      "id": "2",\n      "scene": "The Challenge",\n      "description": "Tortoise challenges Rabbit to a race.",\n      "narration": "The tortoi

MoviePy - Done.
print segments: ['The tortoise, though slow,', "was determined. 'Let's have", "a race!' he challenged."]
Combined audio saved as final_output\Combined_audio\audio_20250409154410\audio_20250409154410.mp3
print segments: ['The race began! The rabbit', 'zoomed ahead, while the', 'tortoise plodded along', 'steadily.']
print segments: ['Confident, the rabbit took a', 'nap under a big tree. The', 'tortoise, however, kept', 'moving.']
print segments: ['When the rabbit woke up,', 'the tortoise had already', 'crossed the finish line! Slow', 'and steady wins the race.']
Total time it takes to convert images into video frames: 7.712367534637451
pre_processing_video saved as final_output\Pre_Generated_videos\pre_video_20250409154410\pre_video_20250409154410.mp4
Merging video and audio
video_path: final_output\Pre_Generated_videos\pre_video_20250409154410\pre_video_20250409154410.mp4, 
audio_path :  ,final_output\Combined_audio\audio_20250409154410\audio_20250409154410.mp3, 
images_

ffmpeg version 2025-03-20-git-76f09ab647-essentials_build-www.gyan.dev Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 14.2.0 (Rev1, Built by MSYS2 project)
  configuration: --enable-gpl --enable-version3 --enable-static --disable-w32threads --disable-autodetect --enable-fontconfig --enable-iconv --enable-gnutls --enable-libxml2 --enable-gmp --enable-bzlib --enable-lzma --enable-zlib --enable-libsrt --enable-libssh --enable-libzmq --enable-avisynth --enable-sdl2 --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxvid --enable-libaom --enable-libopenjpeg --enable-libvpx --enable-mediafoundation --enable-libass --enable-libfreetype --enable-libfribidi --enable-libharfbuzz --enable-libvidstab --enable-libvmaf --enable-libzimg --enable-amf --enable-cuda-llvm --enable-cuvid --enable-dxva2 --enable-d3d11va --enable-d3d12va --enable-ffnvcodec --enable-libvpl --enable-nvdec --enable-nvenc --enable-vaapi --enable-libgme --enable-libopenmpt --enable-libopencore-amrwb -

In [28]:
from typing_extensions import TypedDict
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
import json
import nest_asyncio
from IPython.display import display,Image
from typing import Sequence
import time
import shutil
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, HumanMessage
from typing import Annotated
from datetime import datetime
from langgraph.types import Send
import asyncio
import edge_tts
from moviepy import *
import random
import logging
from ffmpeg import FFmpeg, Progress 
import cv2
import numpy as np
import os
import requests
nest_asyncio.apply()

from dotenv import load_dotenv,find_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
import os
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
   raise ValueError('GEMINI_API_KEY is not set in the env file')
print('GEMINI_API_KEY loaded successfully')    

llm = ChatGoogleGenerativeAI(api_key=GEMINI_API_KEY,model='gemini-1.5-flash')


 
    
class MainCharacters(TypedDict):
    name: str
    appearance: str
    characteristics: str
    
class SupportingCharacters(TypedDict):
    name: str
    appearance: str
    characteristics: str
    
class ScenesList(TypedDict):
    id: str
    scene: str
    description: str
    narration: str
    img_prompt: str
    object_description:str
    save_audio_path:str
    save_image_path:str  
    audio_duration:str

    
class GraphState(TypedDict): 
    story_theme: str
    scene_list: list[ScenesList] 
    supporting_characters: list[SupportingCharacters]
    main_characters:list[MainCharacters] 
    pre_processing_video_path:str
    combined_audio_path:str
    voices_folder:str
    images_folder:str
    messages: Annotated[Sequence[BaseMessage], add_messages]
    
class SubState(TypedDict): 
    current_scene: ScenesList
    output_folder: str
    

def generate_story(state: GraphState) -> GraphState:
    story_theme = state["story_theme"]

    new_prompt = """Based on the given story theme, generate a structured list of  scenes where the total narration duration does not exceed 50 seconds.
    and  create a brief description of the main and supporting character, object, or scene. Include specific details about appearance, characteristics . 
    This description will be used to maintain consistency across multiple scenes.
    Each scene must include an **`Object_Description`** field, which provides a short description of key objects, scenery, or important elements in that scene.  
    and also generate a narration that exactly follows this text, starting with 'Once upon a time...' 

      ### Story Theme:
      {story_theme}

      ### Output Format (JSON):
      {{
        "main_characters": [
              {{
                "name": "",
                "appearance": "",
                "characteristics": ""
              }}
            ],
            "supporting_characters": [
              {{
                "name": "",
                "appearance": "",
                "characteristics": ""
              }}
            ],
          "scenes": [
            {{
              "id": "1",
              "scene": "Engaging Beginning",
              "description": "Begin with a captivating moment to grab children's attention.",
              "narration": ""
              "object_description": ""
            }},
         
          ],
          
        }}

      - **Generate  scenes** to ensure a smooth and structured story.
      - **Don't generate more than 5 scenes.**
      - **Start with an engaging scene** to hook the audience immediately. 
      - **Write concise, engaging, and clear narration** to fit within 50 seconds.
      - **End with a meaningful but natural lesson** without making it feel forced.
      - **Use simple and engaging language** suitable for children.
      - **Ensure smooth transitions** so the story flows naturally.

      Return ONLY valid JSON output without any extra formatting or explanations. Do NOT use markdown formatting (e.g., no triple backticks).
      Strictly output **only JSON** without extra text."""

    sys_msg = SystemMessage(
        content="You are an assistant that extracts structured details from a story."
    )
    hum_msg = HumanMessage(content=new_prompt.format(story_theme=story_theme))

    start_time = time.time()
    res = llm.invoke([sys_msg, hum_msg])
    end_time = time.time()

    print(f"Time taken to generate story characters: {end_time - start_time} seconds")
    print(f"llm result :{res}") 
    raw_content = res.content.strip()
    if raw_content.startswith("```") and raw_content.endswith("```"):
        raw_content = raw_content.strip("`")
        if raw_content.lower().startswith('json'):
            raw_content = raw_content[4:].strip() 
    
    try:
        parsed_resp = json.loads(raw_content)
    except json.JSONDecodeError as e:
        raise ValueError(f"Error parsing JSON: {e}\nRaw content: {raw_content}")

    for scene in parsed_resp["scenes"]:
        prompt_template = f"""Create a detailed, photorealistic image of the following scene:
        {scene["description"]}
        
        **Main Characters**:
        {", ".join([f'{char.get("name",'')} - {char.get("appearance",'')}, {char.get("characteristics",'')}' for char in parsed_resp['main_characters'] if parsed_resp['main_characters']])}
 

        **Supporting Characters**:
        {", ".join([f'{supchar.get("name","")} - {supchar.get("appearance","")} - {supchar.get("characteristics","")}' for supchar in parsed_resp['supporting_characters'] if parsed_resp['supporting_characters']])}
        
        **Objects**:
        {scene["object_description"]}
        **Mood & Lighting**: Cinematic, immersive atmosphere with realistic lighting to match the scene's emotions.

        The illustration should capture the story’s essence and atmosphere."""
        scene["img_prompt"] = prompt_template
 
    state["scene_list"] = parsed_resp["scenes"]
 
    state["supporting_characters"] = parsed_resp.get("supporting_characters", [])
    state["main_characters"] = parsed_resp.get("main_characters", [])
    return state

    
    
async def save_voice(save_path, current_voice):
    await current_voice.save(save_path)


async def call_llm_gen_voice(voice):
    try:
        tts = edge_tts.Communicate(
            voice, voice="en-US-JennyNeural", volume="+100%", pitch="+5Hz"
        )
        return tts
    except Exception as e:
        raise ValueError("Error while generating the voice", e)


async def generate_scene_voice(state: SubState):
    cur_scene = state["current_scene"]
    scene_id = int(cur_scene["id"])
    output_folder = state["output_folder"]
    narration_text = cur_scene.get("narration", "")

    try:
        print(f"Generating voice for scene {scene_id}")
        generated_voice = await call_llm_gen_voice(narration_text)
        save_path = os.path.join(output_folder, f"voice_scene_{scene_id}.mp3")
        save_time = time.time()

        await save_voice(save_path, generated_voice)
        end_save_time = time.time()
        
        print(f"saving time for this scene {scene_id}={end_save_time-save_time}")
        print(f"Generated voice successfully for scene {scene_id}")
        try:
            with AudioFileClip(save_path) as clip:
                audio_duration = clip.duration
        except Exception as e:
            print(f"Error generating voice for scene {scene_id}: {e}")

        message = HumanMessage(
            content="",
            additional_kwargs={
                "voice_id": scene_id,
                "save_path": save_path,
                "audio_duration": audio_duration
            }
        )
        
        return {"messages": [message]}

    except Exception as e:
        print(f"Error generating voice for scene {scene_id}: {e}")


def continue_generate_voice(state: GraphState):
    scene_list = state["scene_list"]
    time_stamp = datetime.now().strftime("%Y%m%d%H%M%S")
    dynamic_folder = f"voices_{time_stamp}"
    output_folder = os.path.join("Generated_voices", dynamic_folder)
    output_folder = os.path.join("final_output", output_folder)
    os.makedirs(output_folder, exist_ok=True)  
    return [
        Send(
            "generate_scene_voice",
            {
                "current_scene": scene,
                "output_folder": output_folder,
            },
        )
        for scene in scene_list
    ]

def sync_generate_scene_voice(state: SubState):
    return asyncio.run(generate_scene_voice(state))






def saveImage(image_content, save_path):
    output_folder = os.path.dirname(save_path)
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    with open(save_path, "wb") as f:
        f.write(image_content)


def images_generates(prompt: str):
    try: 
        color_value1 = format(random.randint(0, 16777215), '06x')  
        color_value2 = format(random.randint(0, 16777215), '06x')
        image_url = f"https://placehold.co/1080x1920/{color_value1}/{color_value2}/png" 
        return image_url 

    except Exception as e:
        logging.error(f"Unexpected error generating image: {e}")
        raise


 

def generate_scence_image(state:SubState):
    cur_scene = state["current_scene"]
    scene_id = int(cur_scene["id"])
    output_folder = state["output_folder"]  
    image_prompt = cur_scene.get("img_prompt", "")
    try:
        print(f"Generating image for scene {scene_id}")
        imageUrl = images_generates(image_prompt)
        if not imageUrl:
            raise ValueError("Generated image URL is empty.")
        imageContent = requests.get(imageUrl).content
        if not imageContent:
            raise ValueError("Failed to retrieve image content.")
        save_path = os.path.join(output_folder, f"image_scene_{scene_id}.png")
        saveImage(imageContent, save_path)
        print(f"Images saved to {save_path}")
        message = HumanMessage(content='',additional_kwargs={"image_id": scene_id,"save_path": save_path})
        return {'messages':[message]}

    except Exception as e:
        logging.error(f"Error processing scene {scene_id}: {e}")
       
 
     

def continue_scence_image(state:GraphState):
    scene_list = state['scene_list']
    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    dynamic_folder = f"images_{timestamp}"
    output_folder = os.path.join("Generated_images", dynamic_folder)
    output_folder = os.path.join("final_output", output_folder)
    os.makedirs(output_folder, exist_ok=True) 
    return [Send("generate_scence_image",{'current_scene':value, 'output_folder':output_folder}) for value in scene_list ]
     
     
     
     
     






def final_node(state: GraphState):
    total_messages = state["messages"]
    print(f"Toatal messages:{total_messages}")
    scene_list = state["scene_list"]
    list_images,list_voices =set(),set()
    for scene in scene_list:
        get_id = int(scene["id"])
     
        for msg in total_messages:
            image_id = msg.additional_kwargs.get('image_id')
            voice_id = msg.additional_kwargs.get("voice_id")
            
            if voice_id is not None and get_id == voice_id:
                aud_dur = msg.additional_kwargs.get('audio_duration','')
                save_path = msg.additional_kwargs.get('save_path','')
                print(f"{get_id} {msg}")
                scene["save_audio_path"] = save_path
                scene["audio_duration"] = aud_dur
                list_voices.add(os.path.dirname(save_path))
            if get_id== image_id : 
                save_path = msg.additional_kwargs.get('save_path','')
                scene['save_image_path'] = msg.additional_kwargs.get('save_path','')
                list_images.add(os.path.dirname(save_path))
    state['images_folder'] =list(list_images)[0]  if list_images else ""      
    state['voices_folder'] =list(list_voices)[0]  if list_voices else ""      
    return state

def zoom_in(image, num_frames=30, zoom_factor=0.5):
    """Generates frames for a zoom-in effect."""
    frames = []
    h, w = image.shape[:2]
    for i in range(num_frames):
        scale = 1.0 + (i / num_frames) * zoom_factor
        center = (w // 2, h // 2)
        M = cv2.getRotationMatrix2D(center, 0, scale)
        frame = cv2.warpAffine(image, M, (w, h))
        frames.append(frame)
    return frames


def zoom_out(image, num_frames=30, zoom_factor=0.5):
    """Generates frames for a zoom-out effect without blinking."""
    frames = []
    h, w = image.shape[:2]
    for i in range(num_frames):
        scale = 1.0 + ((num_frames - i - 1) / num_frames) * zoom_factor
        center = (w // 2, h // 2)
        M = cv2.getRotationMatrix2D(center, 0, scale)
        frame = cv2.warpAffine(image, M, (w, h))
        frames.append(frame)
    return frames


def fade_in(image, num_frames=2):
    """Creates a fade-in effect from black to the image."""
    frames = []
    black = np.zeros_like(image)
    for i in range(num_frames):
        alpha = i / num_frames
        frame = cv2.addWeighted(image, alpha, black, 1 - alpha, 0)
        frames.append(frame)
    return frames


def fade_out(image, num_frames=30):
    """Creates a fade-out effect from image to black."""
    frames = []
    black = np.zeros_like(image)
    for i in range(num_frames):
        alpha = 1 - i / num_frames
        frame = cv2.addWeighted(image, alpha, black, 1 - alpha, 0)
        frames.append(frame)
    return frames
def pre_processing_video(state:GraphState):
    fps=1  ############################ Frame per second ##############
    
    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    dynamic_folder=f"pre_video_{timestamp}"
    output_folder=os.path.join("Pre_Generated_videos",dynamic_folder)
    output_folder=os.path.join("final_output",output_folder)
    os.makedirs(output_folder, exist_ok=True)
    file_name = f'pre_video_{timestamp}.mp4'
    
    pre_processing_video_path=os.path.join(output_folder,file_name)
    scene_list=state['scene_list']
    if not scene_list:
        raise ValueError("No scenes provided.")

    first_img = cv2.imread(scene_list[0]["save_image_path"])
    if first_img is None:
        raise ValueError(f"Cannot load image: {scene_list[0]['save_image_path']}")

    h, w, _ = first_img.shape
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    video_writer = cv2.VideoWriter(pre_processing_video_path, fourcc, fps, (w, h))
    starting_time = time.time()
    for idx, scene in enumerate(scene_list):
        img_path = scene["save_image_path"]
        duration = float(scene["audio_duration"])

        img = cv2.imread(img_path)
        if img is None:
            print(f"Warning: Cannot load image {img_path}. Skipping.")
            continue
        img = cv2.resize(img, (w, h))

        total_frames = int(fps * duration)
        fade_in_frame = int(fps * 1)
        fade_out_frame = int(fps * 1)
        first_image_duration = total_frames - fade_out_frame

        if idx == 0:
            zoom_in_frames = zoom_in(img, num_frames=first_image_duration, zoom_factor=0.5)
            final_zoom_frame = zoom_in_frames[-1]
            fade_out_frames = fade_out(final_zoom_frame, num_frames=fade_out_frame)
            effect_frames = zoom_in_frames + fade_out_frames
        elif idx % 2 == 1:
            zoom_out_frames = zoom_out(img, num_frames=first_image_duration, zoom_factor=0.5)
            final_zoom_frame = zoom_out_frames[-1]
            fade_out_frames = fade_out(final_zoom_frame, num_frames=fade_out_frame)
            effect_frames = zoom_out_frames + fade_out_frames
        else:
            fade_in_frames = fade_in(img, num_frames=fade_in_frame)
            zoom_in_frames = zoom_in(img, num_frames=first_image_duration, zoom_factor=0.5)
            final_zoom_frame = zoom_in_frames[-1]
            fade_out_frames = fade_out(final_zoom_frame, num_frames=fade_out_frame)
            effect_frames = fade_in_frames + zoom_in_frames + fade_out_frames

        for frame in effect_frames:
            text = "Youtube"
            font = cv2.FONT_HERSHEY_SIMPLEX
            font_scale = 1
            font_thickness = 2
            color = (220, 255, 255)
            text_size = cv2.getTextSize(text, font, font_scale, font_thickness)[0]
            text_x = w - text_size[0] - 20
            text_y = 60
            cv2.putText(frame, text, (text_x, text_y), font, font_scale, color, font_thickness, cv2.LINE_AA)

            video_writer.write(frame)

    video_writer.release()
    ending_time = time.time()
    print(f'Total time it takes to convert images into video frames: {ending_time -starting_time}')
    
    print(f"pre_processing_video saved as {pre_processing_video_path}")     
  
    return {'pre_processing_video_path':pre_processing_video_path}
from pydub import AudioSegment
def combining_audio(state:GraphState):
    scene_list=state['scene_list']
    voice_clips = []
    for voice in scene_list:
        print(f'combining _audio {voice}')
        audio_path=voice['save_audio_path']
        if not os.path.exists(audio_path):
            print(f"Warning: Audio file not found:{audio_path}. Skipping")
            continue
        try: 
            voice_clips.append(AudioFileClip(audio_path))
        except Exception as e:
            print(f"Error loading {audio_path}:{e}")    
    if not voice_clips:
        raise ValueError("No Valid audio files to combine")
    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    dynamic_folder=f"audio_{timestamp}"
    output_folder=os.path.join("Combined_audio",dynamic_folder)
    output_folder=os.path.join("final_output",output_folder)
    os.makedirs(output_folder, exist_ok=True)
    file_name = f'audio_{timestamp}.mp3'
    final_path=os.path.join(output_folder,file_name)
    
     
    with concatenate_audioclips(voice_clips) as audio:
        audio.write_audiofile(final_path)
    narration = AudioSegment.from_file(final_path)
    bg_music = AudioSegment.from_file("bg_music.mp3")
    bg_music = bg_music - 45
    while len(bg_music) < len(narration):
        bg_music += bg_music    
    bg_music = bg_music[:len(narration)] 
    merged_audio = narration.overlay(bg_music)
    merged_audio.export(final_path, format="mp3")
    print(f"Combined audio saved as {final_path}")  
    return {'combined_audio_path':final_path}
 

def finaliaz_video(state: GraphState):
    time_stamp = datetime.now().strftime("%Y%m%d%H%M%S")
    video_path = state['pre_processing_video_path']  
    audio_path = state['combined_audio_path']
    images_folder = state['images_folder']
    voices_folder = state['voices_folder']
    
    output_path = f"./final_output/final_video_{time_stamp}.mp4" 
    print('Merging video and audio') 
    print(f'video_path: {video_path}, ')
    print(f'audio_path :  ,{audio_path}, ')
    print(f'images_folder:  {images_folder}')
    print(f'voices_folder : {voices_folder}')
    !ffmpeg -i "{video_path}" -i "{audio_path}" -c:v copy -c:a aac -strict experimental "{output_path}"
    
     
    def delete_file_and_parent(file_path):
        if os.path.exists(file_path):
            try:
                os.remove(file_path)
                print(f"Deleted file: {file_path}")
            except PermissionError as e:
                print(f"PermissionError: {e} for file {file_path}")
            parent_folder = os.path.dirname(file_path)
            if os.path.exists(parent_folder) and not os.listdir(parent_folder):
                try:
                    shutil.rmtree(parent_folder)
                    print(f"Deleted empty folder: {parent_folder}")
                except Exception as e:
                    print(f"Error deleting folder {parent_folder}: {e}")
    
    def delete_folder(folder_path):
        if folder_path and os.path.exists(folder_path):
            try:
                shutil.rmtree(folder_path)
                print(f"Deleted folder: {folder_path}")
            except Exception as e:
                print(f"Error deleting folder {folder_path}: {e}")
    
   
     
    delete_file_and_parent(video_path)
    delete_file_and_parent(audio_path)
    delete_folder(voices_folder)     
    delete_folder(images_folder)
    print("Merged video and audio successfully and output path is ", output_path)
    return state

workflow = StateGraph(GraphState)
workflow.add_node('generate_story',generate_story) 
workflow.add_node("generate_scene_voice", sync_generate_scene_voice)
workflow.add_node('pre_processing_video',pre_processing_video)
workflow.add_node('generate_scence_image',generate_scence_image) 
workflow.add_node("combining_audio", combining_audio) 
workflow.add_node('final_node',final_node)
workflow.add_node('finaliaz_video',finaliaz_video)


workflow.add_edge(START,'generate_story') 
workflow.add_conditional_edges('generate_story', continue_generate_voice, ["generate_scene_voice"])
workflow.add_conditional_edges('generate_story',continue_scence_image,['generate_scence_image'])
workflow.add_edge(["generate_scene_voice","generate_scence_image"], "final_node")
workflow.add_edge('final_node','combining_audio') 
workflow.add_edge("final_node", 'pre_processing_video')
workflow.add_edge(['pre_processing_video','combining_audio'],'finaliaz_video') 
workflow.add_edge('finaliaz_video',END)
 
app = workflow.compile()
 
story_theme =  """Once upon a time, an old man had three sons. As he neared the end of his life, he decided to test their wisdom to determine who was worthy of inheriting his fortune. He gave each of them a single silver coin and told them to buy something that could fill an entire room.

The eldest son bought hay, but it only filled half the room.
The second son bought cotton, but it still wasn’t enough.
The youngest son bought a small candle and, as he lit it, the entire room was filled with light.

Seeing this, the father smiled and said, “Wisdom and knowledge shine brighter than wealth. You, my youngest son, shall inherit my legacy, for you understand that true greatness lies not in material things, but in the ability to bring light into the world.”

And so, the youngest son carried forward his father’s wisdom, bringing prosperity to the family.""" 
resp = app.invoke({'story_theme':story_theme})
print(resp)

GEMINI_API_KEY loaded successfully
Execuation time in story_completion node takes 5.13958740234375
Before generating the story characters
Execution time takes in generating the story plot is :8.560096740722656
After generating the story characters:content='{\n  "MainCharacters": [\n    {\n      "Name": "Old Man Fitzwilliam",\n      "Appearance": "beard as white as snow and eyes as bright as the summer sun",\n      "Characteristics": "knew his time was short, wanted to make sure his legacy went to the wisest son"\n    },\n    {\n      "Name": "Finn",\n      "Appearance": "quiet and thoughtful, with a twinkle in his eye",\n      "Characteristics": "wise, kind, understanding, generous"\n    }\n  ],\n  "SupportingCharacters": [\n    {\n      "Name": "Barnaby",\n      "Appearance": null,\n      "Characteristics": "eldest son, strong but simple fellow, practical"\n    },\n    {\n      "Name": "Edgar",\n      "Appearance": null,\n      "Characteristics": "middle son, clever but a bit sneaky, 

chunk:  66%|██████▌   | 2356/3558 [00:01<00:00, 1617.49it/s, now=None]

Total time it takes to convert images into video frames: 2.1427206993103027
pre_processing_video saved as final_output\Pre_Generated_videos\pre_video_20250402080213\pre_video_20250402080213.mp4


MoviePy - Done.
finaliaz video paths : final_output\Pre_Generated_videos\pre_video_20250402080213\pre_video_20250402080213.mp4,final_output\Combined_audio\audio_20250402080214\audio_20250402080214.mp3,final_output\Generated_images\images_20250402080208,final_output\Generated_voices\voices_20250402080208
Merging video and audio


ffmpeg version 2025-03-20-git-76f09ab647-essentials_build-www.gyan.dev Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 14.2.0 (Rev1, Built by MSYS2 project)
  configuration: --enable-gpl --enable-version3 --enable-static --disable-w32threads --disable-autodetect --enable-fontconfig --enable-iconv --enable-gnutls --enable-libxml2 --enable-gmp --enable-bzlib --enable-lzma --enable-zlib --enable-libsrt --enable-libssh --enable-libzmq --enable-avisynth --enable-sdl2 --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxvid --enable-libaom --enable-libopenjpeg --enable-libvpx --enable-mediafoundation --enable-libass --enable-libfreetype --enable-libfribidi --enable-libharfbuzz --enable-libvidstab --enable-libvmaf --enable-libzimg --enable-amf --enable-cuda-llvm --enable-cuvid --enable-dxva2 --enable-d3d11va --enable-d3d12va --enable-ffnvcodec --enable-libvpl --enable-nvdec --enable-nvenc --enable-vaapi --enable-libgme --enable-libopenmpt --enable-libopencore-amrwb -

Deleted file: final_output\Pre_Generated_videos\pre_video_20250402080213\pre_video_20250402080213.mp4
Deleted empty folder: final_output\Pre_Generated_videos\pre_video_20250402080213
Deleted file: final_output\Combined_audio\audio_20250402080214\audio_20250402080214.mp3
Deleted empty folder: final_output\Combined_audio\audio_20250402080214
Deleted folder: final_output\Generated_images\images_20250402080208


PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'final_output\\Generated_voices\\voices_20250402080208\\voice_scene_1.mp3'